# 回饋迴圈 (Feedback Loop)

## 模組脈絡：校準層的閉環——讓評估結果回頭改進系統

本筆記隸屬 **07-校準與評估**。評估（01）告訴我們「哪裡不好」，但真正的價值在**閉環**：把低分案例收集起來 → 分析失敗模式 → 改進（prompt/few-shot/檢索/微調）→ 再評估。這就是把不可控系統**持續收斂**的工程迴圈。

本筆記示範一個最小可行的回饋迴圈。

## 0. 環境設定

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()

from openai import OpenAI
client = OpenAI()  # 讀取 OPENAI_API_KEY
import os
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")

## 1. 收集帶評分的執行紀錄

每次線上回答都記錄 (input, output, score, feedback)。這裡用模擬資料示範。

In [ ]:
# 模擬：一批帶使用者評分（1-5）與評語的執行紀錄
logs = [
    {"q": "如何重設密碼？", "a": "請至設定頁點選忘記密碼。", "score": 5, "fb": ""},
    {"q": "退貨要多久？", "a": "不確定，請洽客服。", "score": 2, "fb": "沒給出實際天數"},
    {"q": "運費怎麼算？", "a": "運費依重量計算。", "score": 3, "fb": "太籠統"},
]
low = [r for r in logs if r["score"] <= 3]
print(f"低分案例 {len(low)}/{len(logs)} 筆")

## 2. 分析失敗模式

用 LLM 把低分案例聚類成可行動的「失敗模式」——這是把雜亂回饋收斂成改進方向的關鍵步驟。

In [ ]:
import json

def analyze_failures(low_cases):
    payload = json.dumps(low_cases, ensure_ascii=False)
    resp = client.responses.create(
        model=OPENAI_MODEL,
        text={"format": {"type": "json_object"}},
        input=[{"role": "user", "content": (
            "以下是低分客服回答案例，請歸納 1-3 個共通失敗模式並給改進建議，"
            '輸出 JSON：{"patterns": [{"name": "...", "fix": "..."}]}。\n' + payload
        )}],
    )
    return json.loads(resp.output_text)

analysis = analyze_failures(low)
print(json.dumps(analysis, ensure_ascii=False, indent=2))

## 3. 套用改進：把失敗模式寫回 system prompt

最快的改進是把「改進建議」固化進 system prompt（其次才是 few-shot、檢索、微調）。

In [ ]:
fixes = "\n".join(f"- {p['fix']}" for p in analysis.get("patterns", []))
improved_system = (
    "你是電商客服。回答務必具體、可執行。特別注意以下從歷史低分案例學到的規則：\n" + fixes
)

def answer_v2(question):
    resp = client.responses.create(
        model=OPENAI_MODEL,
        input=[{"role": "system", "content": improved_system},
               {"role": "user", "content": question}],
    )
    return resp.output_text

print(answer_v2("退貨要多久？"))  # 應比原本更具體

## 4. 再評估（回到模組 01 的評估流程）

改進後，用 **01-rag-evaluation** 的同一套評估資料與指標重跑，確認分數真的提升——這一步閉合了迴圈。

```
# 概念：對同一 eval_dataset 重跑 Eval(task=answer_v2)，比較改進前後的 Factuality 分數
```

---

## 本章小結

1. **閉環** = 收集評分紀錄 → 分析失敗模式 → 改進 → 再評估。
2. 改進優先序：system prompt → few-shot → 檢索 → 微調（成本遞增）。
3. 「再評估」用模組 01 的同一套指標，確保改進可被量化驗證。
4. 持續迴圈 = 讓不可控系統**持續收斂**，是生產級 LLM 系統的核心維運動作。